# Data Description

The data covers **two years, October 2024 to September 2026**, and ships with this repository as one table, `data/bike_sharing.csv`, with **one row per hour**. It combines three sources:

- the number of bikes rented in each hour, from the trip records that [Capital Bikeshare](https://capitalbikeshare.com/system-data) publishes. Every trip is counted in the hour in which it started;
- the weather in central Washington, D.C. in each hour, from the [Open-Meteo](https://open-meteo.com/) historical weather archive;
- the public holidays of Washington, D.C.

| Column | Source | Description |
| :-- | :-- | :-- |
| `datetime` | | start of the hour, local time in Washington, D.C. |
| `casual` | rentals | rentals by casual riders, who pay per ride or per day |
| `member` | rentals | rentals by members with an annual or monthly membership |
| `rentals` | rentals | total number of rentals in the hour, **the prediction target** |
| `temperature` | weather | air temperature in °C |
| `feels_like` | weather | apparent ("feels like") temperature in °C, which accounts for wind and humidity |
| `humidity` | weather | relative humidity in % |
| `precipitation` | weather | rain and snow in the hour, in mm of water |
| `snowfall` | weather | snowfall in the hour, in cm |
| `wind_speed` | weather | wind speed in km/h |
| `cloud_cover` | weather | share of the sky covered by clouds, in % |
| `weather_code` | weather | type of weather as a [WMO code](https://open-meteo.com/en/docs/historical-weather-api): 0 clear, 1 to 3 partly cloudy to overcast, 51 to 55 drizzle, 61 to 65 rain (light to heavy), 71 to 75 snow (light to heavy) |
| `holiday` | holidays | name of the public holiday on that day, empty on all other days |

Rentals data © Capital Bikeshare, used under the [Capital Bikeshare Data License Agreement](https://capitalbikeshare.com/data-license-agreement). Weather data by [Open-Meteo.com](https://open-meteo.com/), licensed [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/).

In [ ]:
# Import the libraries you need for this project
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import randint, uniform, loguniform

from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, VotingRegressor, StackingRegressor
from sklearn.dummy import DummyRegressor
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score
from xgboost import XGBRegressor
from catboost import CatBoostRegressor

import plotly.express as px

# 1. Set your goal

We want to predict the rentals per hour, so we have a regression task at hand. If we are using the Mean Absolute Error, we can see how many bikes per hour we would be off on average which could be a helpful number to have for restocking the bikes or to keep as extra stock basically. 

The RMSE is in the same unit and penalises big misses more, so if our RMSE is bigger than the MAE by much, then the model makes a few larger mistakes. 

The R2 could be interesting to compare across models and baselines. 

# 2. Get the Data

Load `data/bike_sharing.csv`, and take a first look at its size and columns. Turn `datetime` into a date type, so that you can extract the hour, the weekday or the month from it later.

In [ ]:
def preprocess_data(df):
    # turn datetime to datetime object and split the string to extract more features 
    df['datetime'] = pd.to_datetime(df['datetime'], format='%Y-%m-%d %H:%M')

    df['hour'] = df['datetime'].dt.hour
    df['weekday'] = df['datetime'].dt.weekday #0 means Monday
    df['month'] = df['datetime'].dt.month

    # 5 is saturday and 6 is sunday
    df['is_weekend'] = df['datetime'].dt.weekday >= 5

    # we assume that it is a working day if its not the weekend and it is not a holiday 
    df['working_day'] = (df['datetime'].dt.weekday < 5) & df['holiday'].isna()
    return df 

In [ ]:
df = pd.read_csv("data/bike_sharing.csv")

df.head()


# 3. Split the data into a training and a test set

In [ ]:
df_train, df_test = train_test_split(df, test_size=0.2, random_state=42)

# save the split right away, before any exploration or feature engineering
df_train.to_csv('data/preprocessed_data/df_train.csv', index=False)
df_test.to_csv('data/preprocessed_data/df_test.csv', index=False)

df_train = preprocess_data(df_train)


# 4. Explore the training set 

###  How to rentals change over the day? 

In [ ]:
df_train = df_train.sort_values('datetime')

In [ ]:
px.line(df_train, 
        x='datetime', 
        y='rentals',
        title='Time series - rentals over datetime')

In [ ]:
df_train['year_month'] = df_train['datetime'].dt.to_period('M').dt.to_timestamp()

monthly = df_train.groupby('year_month', as_index=False)['rentals'].mean()

px.line(monthly, x='year_month', y='rentals', markers=True,
        labels={'year_month': 'Month', 'rentals': 'Mean rentals per hour'},
        title='Mean rentals per hour by month')


In [ ]:
df_train['month'] = df_train['datetime'].dt.month
df_train['year'] = df_train['datetime'].dt.year

by_month = df_train.groupby(['year', 'month'], as_index=False)['rentals'].mean()

px.line(by_month, x='month', y='rentals', color='year', markers=True,
        labels={'month': 'Month', 'rentals': 'Mean rentals per hour', 'year': 'Year'},
        title='Mean rentals per hour by calendar month')


### Is it the same pattern on working days, weekends and holidays?

In [ ]:
df_train['day_type'] = np.select(
    [
        df_train['holiday'].notna(),
        df_train['datetime'].dt.weekday >= 5,
    ],
    ['Holiday', 'Weekend'],
    default='Working day',
)

hourly = df_train.groupby(['day_type', 'hour'], as_index=False)['rentals'].mean()

fig = px.line(
    hourly,
    x='hour',
    y='rentals',
    color='day_type',
    markers=True,
    category_orders={'day_type': ['Working day', 'Weekend', 'Holiday']},
    labels={'hour': 'Hour of day', 'rentals': 'Mean rentals per hour', 'day_type': ''},
    title='Mean rentals by hour of day: working days, weekends and holidays',
)
fig.update_xaxes(dtick=2)
fig.update_layout(hovermode='x unified')
fig.show()


> The commute on working days peaks the bike rentals. At 8AM there are around 1500 rentals per hour as opposed to ~500, at 4pm we have ~2000 rentals as opposed to ~1200.  
Other than that, there's fewer rentals on working days in the hours between 10AM and 4PM than on weekends and holidays. 
In the late evening and night hours, the rentals are also fewer. 

### How do the weather columns relate to the rentals?

In [ ]:
weather_cols = ['temperature', 'feels_like', 'humidity', 'precipitation',
                'snowfall', 'wind_speed', 'cloud_cover']

corr = df_train[weather_cols + ['rentals']].corr()
px.imshow(corr, text_auto='.2f', color_continuous_scale='RdBu_r',
          zmin=-1, zmax=1, title='Correlation of weather columns and rentals')


> Temperature and Feels_Like correlates quite a bit positively and linear with the amount of rentals. If it's warmer, people tend to rent bikes more. 
> Be aware! Temperature and Feels_Like carry pretty much the same information, so we could probably get rid of one of these columns. 

> Humidity is correlated negatively with rentals. If there is higher humidity, people tend to rent less bikes.

> Humidity and Cloud cover correlate with 0.27 - makes sense! But I think we should keep both since it isnt too strong.
> Humidity and WInd Speed with -0.24 - makes sense! But I think we should keep both since it isnt too strong.


In [ ]:
sample = 5000
fig = px.scatter(df_train.sample(sample, random_state=0), x='temperature', y='rentals',
                 opacity=0.3, 
                 trendline='lowess', # locally weighted scatterplot smoothing -> smooth curve that goes through the scatter plot to follow the shape of the data
                 title=f'Rentals vs temperature ({sample} sampled hours)')
fig.show()


> Keep in mind: At night it is colder anyways and at night there are less rental! 

In [ ]:
daytime = df_train[
    (df_train['hour'].between(8, 18)) # inclusive on both ends
    & (df_train['day_type'] == 'Working day')
]

fig = px.scatter(
    daytime.sample(min(sample, len(daytime)), random_state=0),
    x='temperature',
    y='rentals',
    color='hour',
    opacity=0.4,
    trendline='lowess',
    trendline_scope='overall',
    labels={'temperature': 'Temperature (°C)', 'rentals': 'Rentals per hour'},
    title=f'Rentals vs temperature, working days, 8:00 to 18:00, {sample=}',
)
fig.show()


> This looks a lot better. At first it looked like very high temperatures would lead to more rentals - if looking at the trend line. Here we can see that until a certain temperature, it seems to lead to more bike rentals but then it plateaus. Which makes sense because most people don't want to ride the bike at above 30 degrees celsius outside.

In [ ]:
df_train['rain'] = pd.cut(df_train['precipitation'], [-0.01, 0, 0.5, 2, 100],
                          labels=['none', 'light', 'moderate', 'heavy'])

px.box(df_train, x='rain', y='rentals',
       category_orders={'rain': ['none', 'light', 'moderate', 'heavy']},
       title='Rentals by precipitation')


> If it rains just a little, it doesn't seem to have an effect on the rentals but moderate or heavy rain seems to lead to less bike rentals.

In [ ]:
df_train.rain.value_counts()

### Which columns contain the same information? 

> Temperature and Feels_Like seem to correlate quite a bit just from looking at our plots! Let's check it. 

In [ ]:
df_train['temperature'].corr(df_train['feels_like'])

> yes! Let's get rid of one of them. Feels_Like does kind of include humidity, so now we could think about whether we wanted "pure" measures or just keep Feels_Like and get rid of the other ones. 

In [ ]:
df_train[['temperature', 'feels_like', 'rentals']].corr(method='spearman')['rentals']


> Temperature seems to be a slightly better indicator for the target, so let's get rid of Feels_Like later on.

### Is it important which holiday it is?

In [ ]:
px.box(df_train, 
       color='holiday',
       y='rentals',
       title='Rentals per specific holiday')

In [ ]:
df_train['is_holiday'] = df_train['holiday'].notna()
df_train.groupby('is_holiday')['datetime'].apply(lambda s: s.dt.date.nunique())

> Over our two years, our 12 holidays have 24 days of the collected 730. So thats probably too few data points to have to take into account a specific holiday. Instead we could just make a flag for the holiday. 

### more weather analysis

> mostly none values! 

In [ ]:
# Snowfall: mostly zero, so bin it
df_train['snow'] = pd.cut(df_train['snowfall'], [-0.01, 0, 0.5, 100],
                          labels=['none', 'light', 'heavy'])
fig = px.box(df_train, x='snow', y='rentals',
             category_orders={'snow': ['none', 'light', 'heavy']},
             title='Rentals by snowfall')
fig.show()

# Weather code: a category, not a number
fig = px.box(df_train, x='weather_code', y='rentals',
             title='Rentals by weather code')
fig.update_xaxes(type='category')
fig.show()


In [ ]:
df_train.weather_code.value_counts()

### Does a column leak the target?

> Just from the data description, I suspect that adding casual + member would give us the exact value of the target, let's check.

In [ ]:
(df_train['casual'] + df_train['member'] == df_train['rentals']).sum() == len(df_train)

> yes! we absolutely need to drop those two columns. 

### check for multicollinearity

In [ ]:
features = ['temperature', 'feels_like', 'humidity', 'precipitation', 'snowfall',
            'wind_speed', 'cloud_cover', 'hour', 'month']

# 1. Correlation matrix
corr = df_train[features].corr(method='spearman')
px.imshow(corr, text_auto='.2f', color_continuous_scale='RdBu_r',
          zmin=-1, zmax=1, title='Correlation between features')

# 2. VIF
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

X = add_constant(df_train[features].dropna())
vif = pd.Series(
    [variance_inflation_factor(X.values, i) for i in range(1, X.shape[1])],
    index=features, name='VIF',
).sort_values(ascending=False)
print(vif)


> Nothing we had not noticed before

In [ ]:
# get rid of feels_like to check how multicollinearity would change
features = ['temperature', 'humidity', 'precipitation', 'snowfall',
            'wind_speed', 'cloud_cover', 'hour', 'month']

# 1. Correlation matrix
corr = df_train[features].corr(method='spearman')
px.imshow(corr, text_auto='.2f', color_continuous_scale='RdBu_r',
          zmin=-1, zmax=1, title='Correlation between features')

# 2. VIF
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

X = add_constant(df_train[features].dropna())
vif = pd.Series(
    [variance_inflation_factor(X.values, i) for i in range(1, X.shape[1])],
    index=features, name='VIF',
).sort_values(ascending=False)
print(vif)


## Summary of the EDA: what matters for feature engineering

**Drop**
- `casual`, `member`: `casual + member == rentals` for every row, so they leak the target.
- `feels_like`: correlation with `temperature` is 0.99 and VIF is about 217 (`temperature` about 212). `temperature` correlates slightly better with rentals (Spearman 0.47 vs 0.43).
- `holiday` (the name): 24 holiday days in total (2 per holiday) is too few to learn each holiday. Replace it with a flag.
- `datetime` itself, after extracting features.

**Keep**
- `humidity`, `wind_speed`, `cloud_cover`: pairwise correlations are weak (0.27 and -0.24), and every VIF is below 1.5 once `feels_like` is excluded.
- `precipitation`, `snowfall`, `weather_code`.

**Calendar features**
- `hour`, `weekday`, `month` (already in `preprocess_data`), plus `is_weekend`, `is_holiday` (`holiday.notna()`) and `working_day`.
- The daily pattern depends on the day type. Working days have commute peaks at about 8:00 and 16:00 to 18:00. Weekends and holidays peak at midday, and holidays look like weekends. So `hour` needs to be combined with the day type.
- Linear model only: `hour` is cyclical (23 is next to 0), so use sin/cos or one-hot `hour`, plus an interaction `hour` × `working_day`. Trees and boosting handle this on their own.

**Weather features**
- Temperature: rentals rise, then plateau above about 25 to 30 °C. This is non-linear, so a linear model needs a bin or a polynomial term. Trees don't.
- Precipitation: light rain has no visible effect, and moderate or heavy rain lowers rentals. A threshold or bin may help the linear model.
- Snowfall: almost always 0 (13,861 none, 115 light, 40 heavy in the training set), so the evidence is thin. A flag `snow > 0` is enough.
- `weather_code` is categorical: one-hot for Ridge, `cat_features` for CatBoost, and consider grouping rare codes.

**Pipeline rules**
- Every manual change to the training data must also be applied to the test set. Put it in `preprocess_data`.
- Columns created only on `df_train` during EDA: `year_month`, `year`, `day_type`, `rain`, `snow`, `is_holiday`. They don't exist in `df_test`.
- Don't use `year` or the raw date as a feature. The model can't extrapolate to other dates.